# socbench quickstart

This notebook runs an end-to-end **smoke** of the socbench SOC-agent benchmark
on a laptop, with **no API keys required**. It uses the always-on `mock`
provider so the full pipeline — index build, stratified sampling, multi-turn
agent loop, scoring, and artifact writing — executes deterministically and for
free.

Steps:

1. Generate a tiny synthetic NetFlow parquet (a stand-in for the real sample
   dataset until one is committed).
2. `socbench build-index` — normalize → eval units → content-addressed index.
3. `socbench run --providers mock` — run the agent loop over the sampled units.
4. Load `summary.json` and plot per-persona F1 by scoring lens.

To run against real models later, install the provider SDKs
(`pip install -e ".[providers]"`), export `OPENAI_API_KEY` /
`ANTHROPIC_API_KEY` / `GOOGLE_API_KEY`, and swap `--providers mock` for
`--providers all` (or a CSV like `openai,anthropic`).

In [ ]:
import json
import re
import subprocess
import sys
from pathlib import Path

# Locate the repo root (the folder containing pyproject.toml) regardless of
# where the kernel started.
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / "pyproject.toml").exists(), "run this notebook from inside the socbench repo"

CONFIG = REPO / "config" / "benchmark_config.yaml"
SAMPLE_PARQUET = REPO / "data" / "sample" / "cic2018-mini.parquet"


def socbench(*args: str) -> str:
    """Invoke the socbench CLI in-process-equivalent via `python -m`.

    Using the module form keeps the notebook working whether or not the
    console script is on PATH. Raises with captured output on failure.
    """
    cmd = [sys.executable, "-m", "socbench.cli", "--log-level", "WARNING", *args]
    proc = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"`{' '.join(args)}` failed:\n{proc.stdout}\n{proc.stderr}")
    return proc.stdout


print("repo:", REPO)

## 1. Generate a synthetic sample dataset

The repo does not yet ship a real sample parquet, so we synthesize a tiny one
that exercises both eval-unit types: benign conversations, single-target
brute-force pairs (→ `pair_timeline`), and a scanner host fanning out to many
destinations (→ `host_egress`). This is a stand-in purely so the notebook is
self-contained; replace `data/sample/cic2018-mini.parquet` with a real slice
when available.

In [ ]:
import random

import polars as pl


def synthesize_netflow(seed: int = 7) -> pl.DataFrame:
    rng = random.Random(seed)
    rows: list[dict] = []
    ts = 1_700_000_000.0

    def flow(src, dst, ts, attack):
        is_tcp = rng.random() < 0.7
        return {
            "src_ip": src, "dst_ip": dst, "ts_start": ts,
            "protocol": "TCP" if is_tcp else "UDP",
            "src_port": rng.randint(1024, 60000),
            "dst_port": rng.choice([80, 443, 22, 53, 8080, 9999]),
            "bytes_in": float(rng.randint(80, 5000)),
            "bytes_out": float(rng.randint(80, 5000)),
            "pkts_in": float(rng.randint(1, 20)),
            "pkts_out": float(rng.randint(1, 20)),
            "tcp_flags": "S" if is_tcp else "",
            "flow_duration_ms": float(rng.randint(1, 60_000)),
            "sampling_rate": 1,
            "Attack": attack,
            "Label": 0 if attack == "benign" else 1,
        }

    # 8 benign pairs
    for i in range(8):
        for _ in range(5):
            rows.append(flow(f"10.0.0.{i+1}", f"10.0.1.{i+1}", ts, "benign"))
            ts += 1.0
    # 2 single-target attack pairs -> pair_timeline
    for i in range(2):
        for _ in range(4):
            rows.append(flow(f"10.0.10.{i+1}", f"10.0.20.{i+1}", ts, "Brute Force"))
            ts += 1.0
    # scanner host: 12 distinct dsts in a tight window -> host_egress (>= K=10)
    for d in range(12):
        for _ in range(2):
            rows.append(flow("10.0.99.99", f"10.0.30.{d+1}", ts, "PortScan"))
            ts += 0.5
    return pl.DataFrame(rows)


SAMPLE_PARQUET.parent.mkdir(parents=True, exist_ok=True)
df = synthesize_netflow()
df.write_parquet(SAMPLE_PARQUET, compression="zstd", statistics=True)
print(f"wrote {len(df)} flows → {SAMPLE_PARQUET.relative_to(REPO)}")

## 2. Build the content-addressed index

`build-index` is idempotent: the same input bytes always produce the same
`dataset_hash`, and a second run is a no-op. We parse the hash out of the CLI
output to use in the next step.

In [ ]:
out = socbench("build-index", "--config", str(CONFIG), "--dataset", "sample", "--rebuild")
print(out.strip())

dataset_hash = re.search(r"dataset_hash=([a-f0-9]+)", out).group(1)
print("\ndataset_hash:", dataset_hash)

## 3. Run the benchmark (mock provider)

This samples eval units (stratified), then runs the multi-turn agent loop
for every `(unit × persona × provider)` rendering. With `--providers mock` it
costs nothing and is fully deterministic. The CLI prints a small JSON summary;
the full artifact tree lands under `runs/<run_id>/`.

In [ ]:
run_out = socbench(
    "run",
    "--config", str(CONFIG),
    "--dataset-hash", dataset_hash,
    "--providers", "mock",
    "--personas", "all",
)
run_info = json.loads(run_out)
run_dir = REPO / run_info["run_dir"]
print(json.dumps(run_info, indent=2))

## 4. Load results and plot

`summary.json` carries the `scoring` block (macro per-flow / per-pair /
per-host F1 per `provider/persona`) and the `cache` block. We plot the three
F1 lenses per persona.

> The mock provider submits a fixed scripted verdict, so the absolute F1 values
> are meaningless — this chart is here to show the *shape* of the output you'd
> read for real providers.

In [ ]:
import matplotlib.pyplot as plt

summary = json.loads((run_dir / "summary.json").read_text())
scoring = summary["scoring"]

labels = list(scoring.keys())
lenses = ["per_flow_f1_macro", "per_pair_f1_macro", "per_host_f1_macro"]
lens_titles = ["per-flow", "per-pair", "per-host"]

x = range(len(labels))
width = 0.25
fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.4), 4))
for i, (lens, title) in enumerate(zip(lenses, lens_titles)):
    vals = [scoring[k][lens] for k in labels]
    ax.bar([p + i * width for p in x], vals, width, label=title)

ax.set_xticks([p + width for p in x])
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("macro F1")
ax.set_ylim(0, 1.05)
ax.set_title(f"F1 by lens — run {run_info['run_id']}")
ax.legend()
fig.tight_layout()
plt.show()

print("cache block:", json.dumps(summary["cache"], indent=2))

## Next steps

- **Slice the results** — open `notebooks/results_explorer.ipynb` and point it
  at this `run_dir` to break results down by stratum, persona, and provider.
- **Run an ablation** — re-run step 3 with `--ablation tools_off` (or
  `playbooks_off`), then `socbench aggregate --dataset-hash <hash>` to compute
  the `tools_off → main` deltas under `ablations/<hash>/<seed>/`.
- **Use real models** — install `.[providers]`, export the API-key env vars,
  and switch `--providers mock` → `--providers all`. The smoke `cost_budget_usd`
  guardrail (default $10) stops the run cleanly if exceeded.